# Graph Data Load for a Cyclist Volume Model for Greater Adelaide

This notebook loads the graph initally set up by `./GRAPH_SETUP.ipynb` and fills in all the columns to be used for regression modelling. The final graph is saved as `./graph_cache/graph_loaded.graphml` and is retrieved by the different models to test different regression techniques. This notebook tries to include a very comprehensive column list for the graph, such that different combinations of columns, and different techniques can be tried for different models, however it can be modified and re-run if need be.

__*Retrieve generated graph*__

See `./GRAPH_SETUP.ipynb` for graph network generation. Graph comes with all infrastructure features from OpenStreetMap, all listed cycle routes, Mesh Blocks and Destination Zones. Mesh blocks and destination zones are loaded into __*each node*__, so MB and DZN-related variables must be calculated for (u,v) of each edge. Depending on the variable, the minimum, maximum, or average of (u,v) should be taken as the final number for that edge.

In [3]:
import osmnx as ox

graphml_path = "./graph_cache/graph_setup.graphml"
G = ox.load_graphml(filepath=graphml_path)
print(f"Loaded graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

# Verify key attributes
nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)
print(f"IS_CYCLE_ROUTE present: {'IS_CYCLE_ROUTE' in edges.columns}")

Loaded graph: 466150 nodes, 893094 edges
IS_CYCLE_ROUTE present: True


## Spatially Calculated Data

__*DNL_DZN*__

Distance to nearest large destination zone (8000+ jobs). These zones include the areas near North Terrace / Rundle / Franklin st, the RAH, between Pirie and Wakefield St, East End. 8000 is an arbitrary number.

__*PD_MB*__

Population density by mesh block

In [4]:
import sys
sys.path.insert(0, '/Users/alexhuynh/Documents/AdelaideCyclistVolumesModel')
from scripts.distance_to_large_dzns import join_dnl_dzn

G = join_dnl_dzn(G)

Found 8 large DZNs (>= 8000 workers)
Found 3758 nodes in large DZNs
DNL_DZN stats: min=0m, max=40452m, mean=12002m
  Edges in large DZNs (dist=0): 6437/893094


In [5]:
# import importlib
# import scripts.population_density_mbs
# importlib.reload(scripts.population_density_mbs)
from scripts.population_density_mbs import join_pd_mb

G = join_pd_mb(G)

PD_MB stats: min=0.00, max=68421.05, mean=1842.16 (per_km2)
  Edges with zero density: 202926/893094


## Census Table data:

__*POW_CTW*__

N people who work in this DZN and cycle to work

__*POW_FE*__

N people who work in this DZN and are fullly engaged in employment or study

__*M_30_40, F_15_25*__

Age demographics - males 30-40 and females 15-25

__*H_3500*__

Households with a THI of > 3500, place of enumeration, MB

In [6]:
from scripts.join_census_cols import join_all
G = join_all(G)

POW_CTW added: 466131 nodes matched
POW_FE added: 466131 nodes matched


/Users/alexhuynh/Documents/AdelaideCyclistVolumesModel/scripts/join_census_cols.py:113: DtypeWarning: Columns (0: MB (UR), 1: M 0-4 years) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Age demographics added: M_30_40=466131 nodes, F_15_25=466131 nodes matched
H_3500 added: 466131 nodes matched


## MBI Classifiers

All Monash classifiers are retrieved via `infrastructure/get_bike_infra.py`.

In [7]:
from scripts.get_bike_infra import enrich_graph_with_monash_classifiers
G = enrich_graph_with_monash_classifiers(G)

## Misc Variables

__*INF_CR*__

Listed cycle routes

In [8]:
# INF_CR = IS_CYCLE_ROUTE (already on edges from cycle_routes enrichment)
nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)
edges['INF_CR'] = edges['IS_CYCLE_ROUTE'].astype(int)
G = ox.graph_from_gdfs(nodes, edges)
print(f"INF_CR added: {edges['INF_CR'].sum()} edges on cycle routes")

INF_CR added: 44929 edges on cycle routes


# Save graph to file

In [9]:
import os
import osmnx as ox

output_dir = "./graph_cache"
os.makedirs(output_dir, exist_ok=True)

graphml_path = os.path.join(output_dir, "graph_loaded.graphml")
ox.save_graphml(G, filepath=graphml_path)
print(f"Graph saved to {graphml_path}")

Graph saved to ./graph_cache/graph_loaded.graphml


# 